# microWakeWord — Nyra IT V3 FULL (66 GB free-disk profile)

This notebook trains the wake word **Nyra**, pronounced in Italian as **"Nira" / "Nìra"**.

The technical model ID is **`nyra_it`**. The visible wake-word name remains **Nyra**.

## V3 changes

- Uses the user's real positive WAV recordings from `nyra_it_real_samples.zip`.
- Real WAV filenames and folder structure do **not** matter: every `.wav` in the ZIP is treated as a positive Nyra sample.
- Keeps real and synthetic positives as **separate feature sets**, so the 30 real recordings cannot disappear statistically inside thousands of synthetic samples.
- Uses **Italian Piper voices** for synthetic positives instead of the English LibriTTS-R generator.
- Generates a **30-sample synthetic preview first** and requires manual approval before full generation/training.
- Keeps synthetic hard confusable negatives for the first real-world version.
- Exports `nyra_it.tflite` and `nyra_it.json`.

## Disk profile

Designed for a Colab runtime with approximately:
- 113 GB total local disk
- ~66 GB initially free
- ~52 GB RAM
- NVIDIA L4 GPU

The notebook deliberately deletes source WAVs once their mmap features have been generated.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                         NYRA IT V3 CONFIG                             ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# Human-facing name vs technical/model name.
WAKE_WORD = "Nyra"
PRONUNCIATION = "Nira"      # Italian target pronunciation: Nì-ra
OUTPUT_NAME = "nyra_it"

AUTHOR = "Nicola"
AUTHOR_WEBSITE = ""

MODE = "generate"
DRIVE_FOLDER = "wakeword_training_nyra_it"

# Real positives
REAL_SAMPLES_ZIP_NAME = "nyra_it_real_samples.zip"
EXPECTED_REAL_SAMPLES = 30

# Synthetic positives.
# V2 used 24k samples from a 904-speaker English generator.
# V3 has only two native Italian Piper voices, so 12k base samples + augmentation
# is a better quality/compute tradeoff than mechanically making 24k near-duplicates.
SYNTHETIC_SAMPLES = 12000

# Manual pronunciation gate.
SYNTHETIC_PREVIEW_SAMPLES = 30
SYNTHETIC_APPROVAL_TEXT = "OK"

# Feature repetitions.
SYNTHETIC_TRAIN_REPETITION = 2
REAL_TRAIN_REPETITION = 40
CONFUSABLE_TRAIN_REPETITION = 2

# Real positives must matter even though there are only ~30 recordings.
# Sampling weights are per feature source, not proportional to file count.
REAL_POSITIVE_WEIGHT = 5.0
SYNTHETIC_POSITIVE_WEIGHT = 7.0

# Synthetic hard negatives for first field version.
# We generate these as Italian text with Italian voices.
CONFUSABLE_WORDS = [
    "Nina",
    "Mira",
    "Nera",
    "Ira",
    "Gira",
    "Tira",
    "Vira",
    "Lira",
    "Mina",
    "Niro",
    "Nura",
    "Siri",
    "Nino",
    "Mila",
]
SAMPLES_PER_CONFUSABLE = 700

# Never intentionally use the final 8 GB of the VM disk.
DISK_RESERVE_GB = 8.0

# Manifest tuning; tune on-device after real-world testing.
PROBABILITY_CUTOFF = 0.70
SLIDING_WINDOW_SIZE = 3
TENSOR_ARENA_SIZE = 50000
TRAINED_LANGUAGES = ["it"]

print(f"Wake word: {WAKE_WORD}")
print(f"Pronunciation target: {PRONUNCIATION} (Italian)")
print(f"Model ID: {OUTPUT_NAME}")
print(f"Expected real positives: ~{EXPECTED_REAL_SAMPLES}")
print(f"Synthetic positives: {SYNTHETIC_SAMPLES}")
print(f"Synthetic preview requiring approval: {SYNTHETIC_PREVIEW_SAMPLES}")
print(f"Positive sampling weights: real={REAL_POSITIVE_WEIGHT}, synthetic={SYNTHETIC_POSITIVE_WEIGHT}")
print(f"Hard negatives: {len(CONFUSABLE_WORDS)} × {SAMPLES_PER_CONFUSABLE}")
print(f"Disk reserve: {DISK_RESERVE_GB:.1f} GB")


Wake word: Nyra
Pronunciation target: Nira (Italian)
Model ID: nyra_it
Expected real positives: ~30
Synthetic positives: 12000
Synthetic preview requiring approval: 30
Positive sampling weights: real=5.0, synthetic=7.0
Hard negatives: 14 × 700
Disk reserve: 8.0 GB


In [ ]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


Mounted at /content/drive
Drive folder: /content/drive/MyDrive/wakeword_training_nyra_it


In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


Installing dependencies...
OK: microwakeword.audio.* imports clean


In [ ]:
# === Runtime sanity + disk guard helpers ===
import os, shutil, subprocess
from pathlib import Path

os.chdir("/content")

def disk_stats(path="/content"):
    total, used, free = shutil.disk_usage(path)
    gb = 1024**3
    return total/gb, used/gb, free/gb

def show_disk(label="disk"):
    total, used, free = disk_stats()
    print(f"[{label}] total={total:.1f} GB used={used:.1f} GB free={free:.1f} GB")
    return free

def require_free(min_free_gb, label="operation"):
    free = show_disk(label)
    if free < min_free_gb:
        raise RuntimeError(
            f"Not enough disk for {label}: {free:.1f} GB free, "
            f"need at least {min_free_gb:.1f} GB."
        )

def dir_gb(path):
    p = Path(path)
    if not p.exists():
        return 0.0
    result = subprocess.run(
        ["du", "-sb", str(p)],
        capture_output=True, text=True, check=True
    )
    return int(result.stdout.split()[0]) / (1024**3)

# Clean only working data for THIS run if Run All is repeated.
for d in [
    "/content/generated_samples",
    "/content/confusable_negatives",
    "/content/generated_augmented_features",
    "/content/confusable_features",
    "/content/negative_datasets",
    f"/content/trained_models/{OUTPUT_NAME}",
]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

os.makedirs("/content/generated_samples", exist_ok=True)
os.makedirs("/content/confusable_negatives", exist_ok=True)
os.makedirs("/content/negative_datasets", exist_ok=True)

initial_free = show_disk("initial")
if initial_free < 55:
    print(
        "WARNING: this notebook was designed around ~66 GB free. "
        "It will continue, but disk guards may stop it before training."
    )


[initial] total=112.6 GB used=47.4 GB free=65.2 GB


In [ ]:
# === Piper sample generator + Italian voices ===
import glob, os, shutil, subprocess, sys, urllib.request, importlib
from pathlib import Path

PIPER_REPO_DIR = "/content/piper"
PIPER_SAMPLE_GENERATOR_DIR = "/content/piper-sample-generator"
VOICE_DIR = "/content/models/it_IT"

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(
    ["apt-get", "-qq", "install", "-y", "espeak-ng", "ffmpeg"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "pip", "setuptools", "wheel", "cython"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "piper-tts", "piper-sample-generator"],
    check=True,
)

if not os.path.exists(PIPER_REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/rhasspy/piper", PIPER_REPO_DIR],
        check=True,
    )

if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/rhasspy/piper-sample-generator",
         PIPER_SAMPLE_GENERATOR_DIR],
        check=True,
    )

# Build Piper monotonic_align extension (kept from the V2 compatibility setup).
PIPER_PYTHON_DIR = f"{PIPER_REPO_DIR}/src/python"
MA_DIR = f"{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align"
MA_IMPORT_DIR = f"{MA_DIR}/monotonic_align"
MA_BUILD_DIR = f"{MA_DIR}/piper_train/vits/monotonic_align"

shutil.rmtree(f"{PIPER_PYTHON_DIR}/build", ignore_errors=True)
shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
shutil.rmtree(f"{MA_DIR}/piper_train", ignore_errors=True)

os.makedirs(MA_IMPORT_DIR, exist_ok=True)
os.makedirs(MA_BUILD_DIR, exist_ok=True)
open(f"{MA_IMPORT_DIR}/__init__.py", "a").close()

subprocess.run(
    f"cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace",
    shell=True,
    check=True,
)

built = next(iter(glob.glob(f"{MA_BUILD_DIR}/core.*")), None)
assert built, "monotonic_align core extension build failed"
shutil.copy2(built, MA_IMPORT_DIR)

for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

importlib.invalidate_caches()

# Native Italian Piper voices.
# Paola = medium quality; Riccardo = x_low. Using both gives at least female/male
# speaker diversity while preserving Italian grapheme-to-phoneme behavior.
ITALIAN_VOICES = {
    "paola": {
        "onnx": "https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx?download=true",
        "json": "https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx.json?download=true",
        "filename": "it_IT-paola-medium.onnx",
    },
    "riccardo": {
        "onnx": "https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx?download=true",
        "json": "https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx.json?download=true",
        "filename": "it_IT-riccardo-x_low.onnx",
    },
}

os.makedirs(VOICE_DIR, exist_ok=True)
ITALIAN_MODEL_PATHS = []

for voice_name, info in ITALIAN_VOICES.items():
    model_path = f"{VOICE_DIR}/{info['filename']}"
    config_path = f"{model_path}.json"

    if not os.path.exists(model_path):
        print(f"Downloading Italian Piper voice: {voice_name}...")
        urllib.request.urlretrieve(info["onnx"], model_path)
    if not os.path.exists(config_path):
        urllib.request.urlretrieve(info["json"], config_path)

    ITALIAN_MODEL_PATHS.append(model_path)

print("Italian Piper models:")
for p in ITALIAN_MODEL_PATHS:
    print(" -", p)

PIPER_ENV = os.environ.copy()
PIPER_ENV["PYTHONPATH"] = ":".join([
    PIPER_PYTHON_DIR,
    PIPER_SAMPLE_GENERATOR_DIR,
    PIPER_ENV.get("PYTHONPATH", ""),
])

test = subprocess.run(
    [sys.executable, "-c",
     "import piper_train; import piper_sample_generator; print('Piper imports OK')"],
    env=PIPER_ENV,
    capture_output=True,
    text=True,
)
print(test.stdout)
if test.returncode != 0:
    print(test.stderr)
    test.check_returncode()

print("Piper + Italian voices ready")


Italian Piper models:
 - /content/models/it_IT/it_IT-paola-medium.onnx
 - /content/models/it_IT/it_IT-riccardo-x_low.onnx
Piper imports OK

Piper + Italian voices ready


In [ ]:
# === Upload + preprocess REAL Nyra positives ===
# Expected input: nyra_it_real_samples.zip
# Filenames/folders inside the ZIP do NOT matter; every .wav is a positive sample.

import os, shutil, zipfile, subprocess, wave, statistics
from pathlib import Path
from google.colab import files

REAL_RAW_DIR = Path("/content/nyra_real_raw")
REAL_DIR = Path("/content/nyra_real_samples")
REAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
REAL_DIR.mkdir(parents=True, exist_ok=True)

drive_zip = Path(DRIVE_DIR) / REAL_SAMPLES_ZIP_NAME
local_zip = Path("/content") / REAL_SAMPLES_ZIP_NAME

if drive_zip.exists():
    zip_path = drive_zip
    print(f"Using ZIP already in Drive: {zip_path}")
elif local_zip.exists():
    zip_path = local_zip
    print(f"Using ZIP already in runtime: {zip_path}")
else:
    print(f"Upload {REAL_SAMPLES_ZIP_NAME} now.")
    uploaded = files.upload()
    assert uploaded, "No file uploaded."
    uploaded_name = next(iter(uploaded))
    uploaded_path = Path("/content") / uploaded_name

    if uploaded_path.suffix.lower() != ".zip":
        raise RuntimeError(f"Expected a ZIP, got: {uploaded_name}")

    # We accept any uploaded ZIP name and normalize it to the project name.
    if uploaded_path != local_zip:
        shutil.move(str(uploaded_path), str(local_zip))
    zip_path = local_zip

    # Keep a copy in Drive so a runtime reset does not require re-uploading it.
    shutil.copy2(zip_path, drive_zip)
    print(f"Copied real-sample ZIP to Drive: {drive_zip}")

shutil.rmtree(REAL_RAW_DIR, ignore_errors=True)
REAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
shutil.rmtree(REAL_DIR, ignore_errors=True)
REAL_DIR.mkdir(parents=True, exist_ok=True)

# Safe ZIP extraction (reject path traversal).
with zipfile.ZipFile(zip_path) as zf:
    root = REAL_RAW_DIR.resolve()
    for member in zf.infolist():
        target = (REAL_RAW_DIR / member.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    zf.extractall(REAL_RAW_DIR)

wav_files = sorted(
    p for p in REAL_RAW_DIR.rglob("*")
    if (
        p.is_file()
        and p.suffix.lower() == ".wav"
        and "__MACOSX" not in p.parts
        and not p.name.startswith("._")
    )
)

assert wav_files, "No WAV files found anywhere inside the ZIP."

print(f"Found {len(wav_files)} real WAV files.")
if len(wav_files) != EXPECTED_REAL_SAMPLES:
    print(
        f"NOTE: expected around {EXPECTED_REAL_SAMPLES}, found {len(wav_files)}. "
        "This is not fatal."
    )

# Convert only format (16 kHz, mono, signed PCM 16-bit).
# Deliberately no loudness normalization: preserve useful real-world variation.
for i, src in enumerate(wav_files, 1):
    dst = REAL_DIR / f"nyra_real_{i:03d}.wav"
    cmd = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(src),
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(dst),
    ]
    subprocess.run(cmd, check=True)

def wav_duration_s(path):
    with wave.open(str(path), "rb") as wf:
        return wf.getnframes() / float(wf.getframerate())

processed = sorted(REAL_DIR.glob("*.wav"))
durations = [wav_duration_s(p) for p in processed]

print(f"Processed real positives: {len(processed)}")
print(
    f"Duration: min={min(durations):.2f}s  "
    f"median={statistics.median(durations):.2f}s  "
    f"max={max(durations):.2f}s"
)

too_short = [p.name for p, d in zip(processed, durations) if d < 0.20]
too_long = [p.name for p, d in zip(processed, durations) if d > 5.0]
if too_short:
    print("WARNING very short clips:", too_short)
if too_long:
    print("WARNING very long clips:", too_long)

print("✓ Real Nyra positives ready")


Using ZIP already in Drive: /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it_real_samples.zip
Found 30 real WAV files.
Processed real positives: 30
Duration: min=0.50s  median=0.68s  max=0.80s
✓ Real Nyra positives ready


In [ ]:
# === Synthetic Italian preview: RATE EVERY SAMPLE 1..5 ===
# 1 = wrong/unusable pronunciation
# 2 = poor
# 3 = acceptable but not ideal
# 4 = good
# 5 = excellent / exactly how Nyra should sound
#
# The notebook automatically selects the best VOICE+CONFIG families afterwards.

import os, shutil, subprocess, sys, json, statistics
from pathlib import Path
from IPython.display import display, Audio, Markdown

SYNTH_PREVIEW_DIR = Path("/content/nyra_it_synthetic_preview")
shutil.rmtree(SYNTH_PREVIEW_DIR, ignore_errors=True)
SYNTH_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# 5 controlled configurations per Italian voice.
# 2 voices × 5 configs × 3 samples = 30 previews.
SYNTH_FAMILIES = [
    {"id": "natural",         "length": 1.00, "noise": 0.667, "noise_w": 0.80},
    {"id": "slightly_fast",   "length": 0.90, "noise": 0.667, "noise_w": 0.80},
    {"id": "slightly_slow",   "length": 1.10, "noise": 0.667, "noise_w": 0.80},
    {"id": "clear_low_noise", "length": 1.00, "noise": 0.55,  "noise_w": 0.70},
    {"id": "expressive",      "length": 1.00, "noise": 0.82,  "noise_w": 0.92},
]

PREVIEW_PER_FAMILY = 3

def generate_controlled_samples(text, model_path, family, count, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m", "piper_sample_generator",
        text,
        "--max-samples", str(count),
        "--model", str(model_path),
        "--length-scales", str(family["length"]),
        "--noise-scales", str(family["noise"]),
        "--noise-scale-ws", str(family["noise_w"]),
        "--output-dir", str(output_dir),
    ]

    proc = subprocess.run(
        cmd,
        env=PIPER_ENV,
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        proc.check_returncode()

ratings = []
sample_number = 0
total_samples = (
    len(ITALIAN_MODEL_PATHS)
    * len(SYNTH_FAMILIES)
    * PREVIEW_PER_FAMILY
)

display(Markdown(
    "# 🔊 Nyra IT — valutazione sintetici\n"
    "Ascolta ogni campione e assegna un voto **da 1 a 5**.\n\n"
    "- **1** = pronuncia sbagliata / inutilizzabile\n"
    "- **2** = scarsa\n"
    "- **3** = accettabile\n"
    "- **4** = buona\n"
    "- **5** = perfetta, è proprio la *Nira* che vogliamo\n\n"
    "Alla fine il notebook sceglierà automaticamente le configurazioni migliori."
))

for model_path in ITALIAN_MODEL_PATHS:
    voice_name = Path(model_path).stem

    for family in SYNTH_FAMILIES:
        fam_dir = SYNTH_PREVIEW_DIR / voice_name / family["id"]
        shutil.rmtree(fam_dir, ignore_errors=True)

        generate_controlled_samples(
            PRONUNCIATION,
            model_path,
            family,
            PREVIEW_PER_FAMILY,
            fam_dir,
        )

        wavs = sorted(fam_dir.glob("*.wav"))

        if len(wavs) < PREVIEW_PER_FAMILY:
            raise RuntimeError(
                f"Expected {PREVIEW_PER_FAMILY} samples, got {len(wavs)} "
                f"for {voice_name}/{family['id']}"
            )

        for wav in wavs[:PREVIEW_PER_FAMILY]:
            sample_number += 1

            display(Markdown(
                f"### Campione {sample_number:02d}/{total_samples}\n"
                f"**Voce:** `{voice_name}`  \n"
                f"**Configurazione:** `{family['id']}`  \n"
                f"`length={family['length']}, "
                f"noise={family['noise']}, "
                f"noise_w={family['noise_w']}`"
            ))

            display(Audio(filename=str(wav)))

            while True:
                raw = input("Voto 1-5: ").strip()

                try:
                    score = int(raw)
                except ValueError:
                    score = 0

                if 1 <= score <= 5:
                    break

                print("Inserisci un numero intero da 1 a 5.")

            ratings.append({
                "sample": sample_number,
                "voice": voice_name,
                "model_path": str(model_path),
                "family_id": family["id"],
                "length": family["length"],
                "noise": family["noise"],
                "noise_w": family["noise_w"],
                "score": score,
                "wav": str(wav),
            })

# Aggregate by voice + configuration
family_stats = []

keys = sorted(
    set((r["voice"], r["family_id"]) for r in ratings)
)

for voice, family_id in keys:
    group = [
        r for r in ratings
        if r["voice"] == voice
        and r["family_id"] == family_id
    ]

    scores = [r["score"] for r in group]
    first = group[0]

    family_stats.append({
        "voice": voice,
        "model_path": first["model_path"],
        "family_id": family_id,
        "length": first["length"],
        "noise": first["noise"],
        "noise_w": first["noise_w"],
        "avg": statistics.mean(scores),
        "min": min(scores),
        "max": max(scores),
        "scores": scores,
    })

family_stats.sort(
    key=lambda x: (x["avg"], x["min"]),
    reverse=True
)

print("\n=== CLASSIFICA CONFIGURAZIONI ===")

for i, s in enumerate(family_stats, 1):
    print(
        f"{i:02d}. {s['voice']} / {s['family_id']}: "
        f"avg={s['avg']:.2f}, "
        f"min={s['min']}, "
        f"scores={s['scores']}"
    )

# Automatic selection:
# Prefer avg >= 4 and no sample below 3.
selected = [
    s for s in family_stats
    if s["avg"] >= 4.0 and s["min"] >= 3
]

# If too few survive, allow avg >= 3.5.
if len(selected) < 3:
    fallback = [
        s for s in family_stats
        if s["avg"] >= 3.5
    ]
    selected = fallback[:3]

if len(selected) < 2:
    raise RuntimeError(
        "Too few reliable synthetic configurations. "
        "Do not continue training."
    )

# At most 5 families: quality > quantity
SELECTED_SYNTH_FAMILIES = selected[:5]

print("\n=== CONFIGURAZIONI SCELTE AUTOMATICAMENTE ===")

for s in SELECTED_SYNTH_FAMILIES:
    print(
        f"✓ {s['voice']} / {s['family_id']} "
        f"(avg={s['avg']:.2f}, min={s['min']})"
    )

# Save ratings and selection
ratings_path = Path("/content/nyra_it_synthetic_ratings.json")
selection_path = Path("/content/nyra_it_selected_synth_families.json")

ratings_path.write_text(
    json.dumps(ratings, indent=2),
    encoding="utf-8"
)

selection_path.write_text(
    json.dumps(SELECTED_SYNTH_FAMILIES, indent=2),
    encoding="utf-8"
)

shutil.copy2(
    ratings_path,
    Path(DRIVE_DIR) / ratings_path.name
)

shutil.copy2(
    selection_path,
    Path(DRIVE_DIR) / selection_path.name
)

APPROVAL_FILE = Path(
    "/content/.nyra_it_synthetic_pronunciation_approved"
)

APPROVAL_FILE.write_text(
    "rated_and_auto_selected\n",
    encoding="utf-8"
)

print("\n✓ Valutazione completata.")
print("✓ Configurazioni migliori selezionate automaticamente.")
print("✓ Voti e selezione salvati su Google Drive.")

# 🔊 Nyra IT — valutazione sintetici
Ascolta ogni campione e assegna un voto **da 1 a 5**.

- **1** = pronuncia sbagliata / inutilizzabile
- **2** = scarsa
- **3** = accettabile
- **4** = buona
- **5** = perfetta, è proprio la *Nira* che vogliamo

Alla fine il notebook sceglierà automaticamente le configurazioni migliori.

### Campione 01/30
**Voce:** `it_IT-paola-medium`  
**Configurazione:** `natural`  
`length=1.0, noise=0.667, noise_w=0.8`

Voto 1-5: 4


### Campione 02/30
**Voce:** `it_IT-paola-medium`  
**Configurazione:** `natural`  
`length=1.0, noise=0.667, noise_w=0.8`

Voto 1-5: 4


...

Voto 1-5: 5


### Campione 29/30
**Voce:** `it_IT-riccardo-x_low`  
**Configurazione:** `expressive`  
`length=1.0, noise=0.82, noise_w=0.92`

Voto 1-5: 5


### Campione 30/30
**Voce:** `it_IT-riccardo-x_low`  
**Configurazione:** `expressive`  
`length=1.0, noise=0.82, noise_w=0.92`

Voto 1-5: 5

=== CLASSIFICA CONFIGURAZIONI ===
01. it_IT-riccardo-x_low / clear_low_noise: avg=5.00, min=5, scores=[5, 5, 5]
02. it_IT-riccardo-x_low / expressive: avg=5.00, min=5, scores=[5, 5, 5]
03. it_IT-riccardo-x_low / natural: avg=5.00, min=5, scores=[5, 5, 5]
04. it_IT-riccardo-x_low / slightly_slow: avg=5.00, min=5, scores=[5, 5, 5]
05. it_IT-paola-medium / slightly_fast: avg=4.67, min=4, scores=[4, 5, 5]
06. it_IT-paola-medium / slightly_slow: avg=4.67, min=4, scores=[5, 5, 4]
07. it_IT-paola-medium / clear_low_noise: avg=4.33, min=4, scores=[4, 4, 5]
08. it_IT-riccardo-x_low / slightly_fast: avg=4.33, min=3, scores=[3, 5, 5]
09. it_IT-paola-medium / expressive: avg=4.00, min=3, scores=[4, 5, 3]
10. it_IT-paola-medium / natural: avg=3.67, min=3, scores=[4, 4, 3]

=== CONFIGURAZIONI SCELTE AUTOMATICAMENTE ===
✓ it_IT-riccardo-x_low / clear_low_noise (avg=5.00, min=5)
✓ it_IT-riccardo-x_low / expressive (avg=5.00, min=5)
✓ it_IT-riccardo-x_low / natural (avg=5.00, min=5)
✓ it_I

In [ ]:
# === Generate FULL synthetic Italian positive dataset
#     from APPROVED families only ===

import os, shutil, json, math
from pathlib import Path

APPROVAL_FILE = Path(
    "/content/.nyra_it_synthetic_pronunciation_approved"
)

SELECTION_FILE = Path(
    "/content/nyra_it_selected_synth_families.json"
)

assert APPROVAL_FILE.exists(), (
    "Synthetic samples have not been rated yet."
)

assert SELECTION_FILE.exists(), (
    "No selected synthetic families found."
)

SELECTED_SYNTH_FAMILIES = json.loads(
    SELECTION_FILE.read_text(encoding="utf-8")
)

SYNTHETIC_DIR = Path("/content/generated_samples")

shutil.rmtree(
    SYNTHETIC_DIR,
    ignore_errors=True
)

SYNTHETIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

require_free(
    DISK_RESERVE_GB + 4,
    "synthetic positive generation"
)

print(
    "Generating synthetic dataset ONLY "
    "from approved configurations:"
)

for s in SELECTED_SYNTH_FAMILIES:
    print(
        f"  ✓ {s['voice']} / {s['family_id']} "
        f"(avg={s['avg']:.2f}, min={s['min']})"
    )

n_families = len(
    SELECTED_SYNTH_FAMILIES
)

per_family = math.ceil(
    SYNTHETIC_SAMPLES / n_families
)

generated_total = 0

for idx, s in enumerate(
    SELECTED_SYNTH_FAMILIES,
    1
):
    remaining = (
        SYNTHETIC_SAMPLES
        - generated_total
    )

    if remaining <= 0:
        break

    count = min(
        per_family,
        remaining
    )

    out_dir = (
        SYNTHETIC_DIR
        / f"{idx:02d}_{s['voice']}_{s['family_id']}"
    )

    family = {
        "length": s["length"],
        "noise": s["noise"],
        "noise_w": s["noise_w"],
    }

    print(
        f"[{idx}/{n_families}] "
        f"{s['voice']} / {s['family_id']} "
        f"-> {count} samples"
    )

    generate_controlled_samples(
        PRONUNCIATION,
        s["model_path"],
        family,
        count,
        out_dir,
    )

    n = len(
        list(out_dir.glob("*.wav"))
    )

    generated_total += n

    print(
        f"  generated: {n}; "
        f"cumulative={generated_total}"
    )

print(
    f"\nTotal synthetic positive samples: "
    f"{generated_total}"
)

print(
    f"Synthetic positive WAV size: "
    f"{dir_gb(str(SYNTHETIC_DIR)):.2f} GB"
)

show_disk(
    "after synthetic positive WAVs"
)

assert (
    generated_total
    >= SYNTHETIC_SAMPLES * 0.95
), "Too few synthetic positive samples generated"

# Flatten because the next feature-extraction cell
# expects WAVs directly in /content/generated_samples
flat_dir = Path(
    "/content/generated_samples_flat"
)

shutil.rmtree(
    flat_dir,
    ignore_errors=True
)

flat_dir.mkdir(
    parents=True,
    exist_ok=True
)

counter = 0

for wav in SYNTHETIC_DIR.rglob("*.wav"):
    counter += 1

    shutil.move(
        str(wav),
        str(
            flat_dir
            / f"nyra_synth_{counter:06d}.wav"
        )
    )

shutil.rmtree(
    SYNTHETIC_DIR,
    ignore_errors=True
)

shutil.move(
    str(flat_dir),
    str(SYNTHETIC_DIR)
)

print(
    f"Flattened synthetic WAVs: "
    f"{counter}"
)

shutil.rmtree(
    "/content/nyra_it_synthetic_preview",
    ignore_errors=True
)

[synthetic positive generation] total=112.6 GB used=47.8 GB free=64.8 GB
Generating synthetic dataset ONLY from approved configurations:
  ✓ it_IT-riccardo-x_low / clear_low_noise (avg=5.00, min=5)
  ✓ it_IT-riccardo-x_low / expressive (avg=5.00, min=5)
  ✓ it_IT-riccardo-x_low / natural (avg=5.00, min=5)
  ✓ it_IT-riccardo-x_low / slightly_slow (avg=5.00, min=5)
  ✓ it_IT-paola-medium / slightly_fast (avg=4.67, min=4)
[1/5] it_IT-riccardo-x_low / clear_low_noise -> 2400 samples
  generated: 2400; cumulative=2400
[2/5] it_IT-riccardo-x_low / expressive -> 2400 samples
  generated: 2400; cumulative=4800
[3/5] it_IT-riccardo-x_low / natural -> 2400 samples
  generated: 2400; cumulative=7200
[4/5] it_IT-riccardo-x_low / slightly_slow -> 2400 samples
  generated: 2400; cumulative=9600
[5/5] it_IT-paola-medium / slightly_fast -> 2400 samples
  generated: 2400; cumulative=12000

Total synthetic positive samples: 12000
Synthetic positive WAV size: 0.25 GB
[after synthetic positive WAVs] total

In [ ]:
# === Generate synthetic hard confusable negatives (Italian voices) ===
import os, re, shutil
from pathlib import Path

CONFUSABLE_DIR = Path("/content/confusable_negatives")
shutil.rmtree(CONFUSABLE_DIR, ignore_errors=True)
CONFUSABLE_DIR.mkdir(parents=True, exist_ok=True)

for i, word in enumerate(CONFUSABLE_WORDS, 1):
    require_free(DISK_RESERVE_GB + 2, f"confusable {word}")

    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", word.lower()).strip("_")
    tmp = Path(f"/tmp/confusable_{safe}")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)

    print(
        f"[{i}/{len(CONFUSABLE_WORDS)}] "
        f"{word!r}: {SAMPLES_PER_CONFUSABLE} Italian synthetic samples"
    )

    run_piper_text(
        word,
        SAMPLES_PER_CONFUSABLE,
        tmp,
    )

    moved = 0
    for filename in tmp.glob("*.wav"):
        shutil.move(
            str(filename),
            str(CONFUSABLE_DIR / f"{safe}_{filename.name}"),
        )
        moved += 1

    shutil.rmtree(tmp, ignore_errors=True)
    print(f"  ✓ moved {moved}")

n = len(list(CONFUSABLE_DIR.glob("*.wav")))
print(f"Total hard-negative WAVs: {n}")
print(f"Hard-negative WAV size: {dir_gb(str(CONFUSABLE_DIR)):.2f} GB")
show_disk("after confusable WAVs")


[confusable Nina] total=112.6 GB used=48.1 GB free=64.5 GB
[1/14] 'Nina': 700 Italian synthetic samples
  ✓ moved 700
[confusable Mira] total=112.6 GB used=48.1 GB free=64.5 GB
[2/14] 'Mira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Nera] total=112.6 GB used=48.2 GB free=64.5 GB
[3/14] 'Nera': 700 Italian synthetic samples
  ✓ moved 700
[confusable Ira] total=112.6 GB used=48.2 GB free=64.5 GB
[4/14] 'Ira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Gira] total=112.6 GB used=48.2 GB free=64.4 GB
[5/14] 'Gira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Tira] total=112.6 GB used=48.2 GB free=64.4 GB
[6/14] 'Tira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Vira] total=112.6 GB used=48.2 GB free=64.4 GB
[7/14] 'Vira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Lira] total=112.6 GB used=48.2 GB free=64.4 GB
[8/14] 'Lira': 700 Italian synthetic samples
  ✓ moved 700
[confusable Mina] total=112.6 GB used=48.2 GB free=64.4 GB

64.28211975097656

In [ ]:
# === Standard negative FEATURE datasets: sequential + guarded ===
# These are pre-generated mmap/spectrogram feature datasets from kahrendt/microwakeword.
# We download ONE ZIP at a time, extract it, then immediately delete the ZIP.
#
# Required for FULL run:
#   speech
#   no_speech
#   dinner_party
#   dinner_party_eval
#
# The disk guard prevents filling the 113 GB Colab filesystem.
import os, zipfile, shutil
from huggingface_hub import hf_hub_download

BASE = "/content/negative_datasets"
REPO = "kahrendt/microwakeword"
STANDARD_DATASETS = [
    "speech",
    "no_speech",
    "dinner_party",
    "dinner_party_eval",
]

os.makedirs(BASE, exist_ok=True)

def download_extract_feature_dataset(name):
    out_dir = f"{BASE}/{name}"

    if os.path.exists(out_dir) and os.listdir(out_dir):
        print(f"{name}: already present ({dir_gb(out_dir):.2f} GB)")
        return

    # Keep a healthy buffer before we even start the download.
    require_free(DISK_RESERVE_GB + 8, f"download {name}")

    zip_name = f"{name}.zip"
    print(f"\n=== {name}: downloading {zip_name} ===")

    zip_path = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=zip_name,
        local_dir=BASE,
    )

    zip_size = os.path.getsize(zip_path) / (1024**3)
    print(f"ZIP size: {zip_size:.2f} GB")
    show_disk(f"{name} downloaded")

    # Need at least reserve + ZIP size again as a conservative extraction buffer.
    require_free(
        DISK_RESERVE_GB + max(4.0, zip_size * 1.5),
        f"extract {name}",
    )

    print(f"Extracting {name}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(BASE)

    # Delete archive immediately.
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # HF may leave cache blobs; remove only the local download cache for this BASE.
    cache_dir = os.path.join(BASE, ".cache")
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)

    assert os.path.exists(out_dir), f"{name} extraction did not create {out_dir}"

    print(f"✓ {name}: {dir_gb(out_dir):.2f} GB extracted")
    show_disk(f"after {name}")

    # Hard stop while there is still enough space to recover cleanly.
    if disk_stats()[2] < DISK_RESERVE_GB:
        raise RuntimeError(
            f"Disk reserve violated after {name}; stopped before further writes."
        )

for name in STANDARD_DATASETS:
    download_extract_feature_dataset(name)

print("\nStandard negative feature sets ready:")
for name in STANDARD_DATASETS:
    print(f"  {name:18s} {dir_gb(f'{BASE}/{name}'):.2f} GB")

show_disk("standard negatives complete")


[download speech] total=112.6 GB used=48.3 GB free=64.3 GB

=== speech: downloading speech.zip ===


speech.zip: reconstructing file:   0%|          |  0.00B / 3.18GB            

speech.zip: downloading bytes:           |  0.00B            

ZIP size: 2.96 GB
[speech downloaded] total=112.6 GB used=51.3 GB free=61.3 GB
[extract speech] total=112.6 GB used=51.3 GB free=61.3 GB
Extracting speech...
✓ speech: 13.08 GB extracted
[after speech] total=112.6 GB used=61.4 GB free=51.2 GB
[download no_speech] total=112.6 GB used=61.4 GB free=51.2 GB

=== no_speech: downloading no_speech.zip ===


no_speech.zip: reconstructing file:   0%|          |  0.00B / 2.00GB            

...

ZIP size: 0.41 GB
[dinner_party downloaded] total=112.6 GB used=70.7 GB free=41.9 GB
[extract dinner_party] total=112.6 GB used=70.7 GB free=41.9 GB
Extracting dinner_party...
✓ dinner_party: 2.18 GB extracted
[after dinner_party] total=112.6 GB used=72.5 GB free=40.1 GB
[download dinner_party_eval] total=112.6 GB used=72.5 GB free=40.1 GB

=== dinner_party_eval: downloading dinner_party_eval.zip ===


dinner_party_eval.zip: reconstructing file:   0%|          |  0.00B / 82.3MB            

dinner_party_eval.zip: downloading bytes:           |  0.00B            

ZIP size: 0.08 GB
[dinner_party_eval downloaded] total=112.6 GB used=72.6 GB free=40.0 GB
[extract dinner_party_eval] total=112.6 GB used=72.6 GB free=40.0 GB
Extracting dinner_party_eval...
✓ dinner_party_eval: 0.40 GB extracted
[after dinner_party_eval] total=112.6 GB used=72.9 GB free=39.7 GB

Standard negative feature sets ready:
  speech             13.08 GB
  no_speech          8.90 GB
  dinner_party       2.18 GB
  dinner_party_eval  0.40 GB
[standard negatives complete] total=112.6 GB used=72.9 GB free=39.7 GB


39.71919631958008

In [ ]:
# === Real + synthetic positive + confusable feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os, shutil, traceback

# Rich self-contained augmentation.
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.20,
        "TanhDistortion": 0.10,
        "PitchShift": 0.20,
        "BandStopFilter": 0.10,
        "AddColorNoise": 0.35,
        "Gain": 1.00,
        "GainTransition": 0.30,
    },
    impulse_paths=[],
    background_paths=[],
    background_min_snr_db=-5,
    background_max_snr_db=20,
    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

def generate_feature_set(input_dir, output_root, label, training_repetition):
    split_config = {
        "training": {
            "split_name": "train",
            "repetition": training_repetition,
            "slide_frames": 10,
        },
        "validation": {
            "split_name": "validation",
            "repetition": 1,
            "slide_frames": 10,
        },
        "testing": {
            "split_name": "test",
            "repetition": 1,
            "slide_frames": 1,
        },
    }

    clips = Clips(
        input_directory=input_dir,
        file_pattern="*.wav",
        max_clip_duration_s=None,
        remove_silence=True,
        random_split_seed=42,
        split_count=0.1,
    )

    os.makedirs(output_root, exist_ok=True)

    for split, cfg in split_config.items():
        require_free(DISK_RESERVE_GB + 3, f"{label} {split} features")

        out = f"{output_root}/{split}"
        mmap = f"{out}/wakeword_mmap"

        if os.path.exists(mmap):
            shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)

        print(
            f"Generating {label} {split} "
            f"(rep={cfg['repetition']}, slide={cfg['slide_frames']})..."
        )

        try:
            sg = SpectrogramGeneration(
                clips=clips,
                augmenter=augmenter,
                slide_frames=cfg["slide_frames"],
                step_ms=10,
            )

            RaggedMmap.from_generator(
                out_dir=mmap,
                batch_size=200,
                verbose=True,
                sample_generator=sg.spectrogram_generator(
                    split=cfg["split_name"],
                    repeat=cfg["repetition"],
                ),
            )
        except Exception:
            traceback.print_exc()
            shutil.rmtree(mmap, ignore_errors=True)
            raise

        print(
            f"  {label}/{split}: "
            f"{dir_gb(out):.2f} GB; free={disk_stats()[2]:.1f} GB"
        )

# 1) REAL positives: only ~30 originals, deliberately augmented many times.
generate_feature_set(
    "/content/nyra_real_samples",
    "/content/real_positive_features",
    "REAL positive",
    REAL_TRAIN_REPETITION,
)
print(f"✓ real positive features: {dir_gb('/content/real_positive_features'):.2f} GB")

# Real ZIP is already in Drive, so runtime copies can now be reclaimed.
shutil.rmtree("/content/nyra_real_samples", ignore_errors=True)
shutil.rmtree("/content/nyra_real_raw", ignore_errors=True)
print("Deleted runtime real WAV copies")
show_disk("after real WAV cleanup")

# 2) Synthetic Italian positives.
generate_feature_set(
    "/content/generated_samples",
    "/content/synthetic_positive_features",
    "synthetic positive",
    SYNTHETIC_TRAIN_REPETITION,
)
print(f"✓ synthetic positive features: {dir_gb('/content/synthetic_positive_features'):.2f} GB")

shutil.rmtree("/content/generated_samples", ignore_errors=True)
print("Deleted synthetic source WAVs")
show_disk("after synthetic WAV cleanup")

# 3) Hard confusables.
generate_feature_set(
    "/content/confusable_negatives",
    "/content/confusable_features",
    "confusable",
    CONFUSABLE_TRAIN_REPETITION,
)
print(f"✓ confusable features: {dir_gb('/content/confusable_features'):.2f} GB")

shutil.rmtree("/content/confusable_negatives", ignore_errors=True)
print("Deleted confusable source WAVs")

print("\nFeature extraction complete:")
for p in [
    "/content/real_positive_features",
    "/content/synthetic_positive_features",
    "/content/confusable_features",
    "/content/negative_datasets",
]:
    print(f"  {p}: {dir_gb(p):.2f} GB")
show_disk("before training config")

if disk_stats()[2] < DISK_RESERVE_GB:
    raise RuntimeError("Disk reserve violated before training")


[REAL positive training features] total=112.6 GB used=72.9 GB free=39.7 GB
Generating REAL positive training (rep=40, slide=10)...


0it [00:00, ?it/s]

  REAL positive/training: 0.44 GB; free=39.3 GB
[REAL positive validation features] total=112.6 GB used=73.4 GB free=39.3 GB
Generating REAL positive validation (rep=1, slide=10)...


0it [00:00, ?it/s]

  REAL positive/validation: 0.00 GB; free=39.3 GB
[REAL positive testing features] total=112.6 GB used=73.4 GB free=39.3 GB
Generating REAL positive testing (rep=1, slide=1)...


...

  confusable/training: 7.21 GB; free=22.9 GB
[confusable validation features] total=112.6 GB used=89.7 GB free=22.9 GB
Generating confusable validation (rep=1, slide=10)...


0it [00:00, ?it/s]

  confusable/validation: 0.45 GB; free=22.4 GB
[confusable testing features] total=112.6 GB used=90.2 GB free=22.4 GB
Generating confusable testing (rep=1, slide=1)...


0it [00:00, ?it/s]

  confusable/testing: 0.05 GB; free=22.4 GB
✓ confusable features: 7.70 GB
Deleted confusable source WAVs

Feature extraction complete:
  /content/real_positive_features: 0.44 GB
  /content/synthetic_positive_features: 9.43 GB
  /content/confusable_features: 7.70 GB
  /content/negative_datasets: 24.56 GB
[before training config] total=112.6 GB used=90.0 GB free=22.6 GB


In [ ]:
# === FULL V3 training config YAML ===
import yaml, os
from pathlib import Path

required_paths = [
    "/content/real_positive_features/training/wakeword_mmap",
    "/content/synthetic_positive_features/training/wakeword_mmap",
    "/content/confusable_features/training/wakeword_mmap",
    "/content/negative_datasets/speech",
    "/content/negative_datasets/no_speech",
    "/content/negative_datasets/dinner_party",
    "/content/negative_datasets/dinner_party_eval",
]
for p in required_paths:
    assert Path(p).exists(), f"Required feature set missing: {p}"

config = {
    "window_step_ms": 10,
    "train_dir": f"/content/trained_models/{OUTPUT_NAME}",

    "features": [
        # The 30 real recordings stay separate and receive a large explicit share
        # of positive sampling instead of being drowned by synthetic file count.
        dict(
            features_dir="/content/real_positive_features",
            sampling_weight=REAL_POSITIVE_WEIGHT,
            penalty_weight=2.0,
            truth=True,
            truncation_strategy="truncate_start",
            type="mmap",
        ),
        dict(
            features_dir="/content/synthetic_positive_features",
            sampling_weight=SYNTHETIC_POSITIVE_WEIGHT,
            penalty_weight=2.0,
            truth=True,
            truncation_strategy="truncate_start",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/speech",
            sampling_weight=10.0,
            penalty_weight=2.5,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/dinner_party",
            sampling_weight=15.0,
            penalty_weight=3.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/no_speech",
            sampling_weight=5.0,
            penalty_weight=1.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
        # Evaluation only.
        dict(
            features_dir="/content/negative_datasets/dinner_party_eval",
            sampling_weight=0.0,
            penalty_weight=1.0,
            truth=False,
            truncation_strategy="split",
            type="mmap",
        ),
        dict(
            features_dir="/content/confusable_features",
            sampling_weight=10.0,
            penalty_weight=6.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
    ],

    "training_steps": [25000, 20000],
    "positive_class_weight": [2, 2],
    "negative_class_weight": [40, 50],
    "learning_rates": [0.001, 0.0001],

    "batch_size": 128,

    "time_mask_max_size": [5, 5],
    "time_mask_count": [1, 1],
    "freq_mask_max_size": [3, 3],
    "freq_mask_count": [1, 1],

    "eval_step_interval": 500,
    "clip_duration_ms": 1500,

    "target_minimization": 0.4,
    "minimization_metric": "ambient_false_positives_per_hour",
    "maximization_metric": "average_viable_recall",
}

os.makedirs(f"/content/trained_models/{OUTPUT_NAME}", exist_ok=True)

with open("/content/training_parameters.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("FULL Nyra IT V3 training_parameters.yaml ready")
print(f"Feature sets: {len(config['features'])}")
print(f"Total steps: {sum(config['training_steps'])}")
print(f"Batch size: {config['batch_size']}")
print(
    "Positive source weights: "
    f"real={REAL_POSITIVE_WEIGHT}, synthetic={SYNTHETIC_POSITIVE_WEIGHT}"
)
show_disk("training config ready")


FULL Nyra IT V3 training_parameters.yaml ready
Feature sets: 7
Total steps: 45000
Batch size: 128
Positive source weights: real=5.0, synthetic=7.0
[training config ready] total=112.6 GB used=90.0 GB free=22.6 GB


22.605506896972656

In [ ]:
# === Train the FULL model ===
import os, sys, subprocess, shutil

require_free(DISK_RESERVE_GB, "training start")

train_dir = f"/content/trained_models/{OUTPUT_NAME}"
shutil.rmtree(train_dir, ignore_errors=True)

env = os.environ.copy()
env["PYTHONPATH"] = "/content/microWakeWord:" + env.get("PYTHONPATH", "")
env["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"

cmd = [
    sys.executable,
    "-m", "microwakeword.model_train_eval",
    "--training_config", "/content/training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

print("Running FULL training:")
print(" ".join(cmd))
print()
show_disk("training launch")

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print()
print("Exit code:", proc.returncode)

assert proc.returncode == 0, (
    "Training failed. Do NOT regenerate feature data; inspect the output above."
)

print("✓ FULL training completed")
show_disk("after training")


Output streaming troncato alle ultime 5000 righe.
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.993; Loss = 0.0233; Mini-Batch #360
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.993; Loss = 0.0233; Mini-Batch #361
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.993; Loss = 0.0233; Mini-Batch #362
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.993; Loss = 0.0233; Mini-Batch #363
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.993; Loss = 0.0233; Mini-Batch #364
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.994; Loss = 0.0233; Mini-Batch #365
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.994; Loss = 0.0233; Mini-Batch #366
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.994; Loss = 0.0233; Mini-Batch #367
Validation Batch #81: Accuracy = 0.993; Recall = 0.976; Precision = 0.994; Loss = 0.0233; Mini-Batch #

22.55337142944336

In [ ]:
# === Verify final model ===
import os

root = f"/content/trained_models/{OUTPUT_NAME}"

os.system(
    f'find "{root}" -type f '
    r'\( -name "*.tflite" -o -name "*.h5" -o -name "*.keras" -o -name "*.ckpt*" \) '
    '-exec ls -lh {} \\; | tail -30'
)

tflite_src = (
    f"{root}/tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

assert os.path.exists(tflite_src), (
    f"Final quantized TFLite not found at {tflite_src}"
)

print()
print("✓ Final quantized model found")
print(f"Size: {os.path.getsize(tflite_src) / 1024:.1f} KB")



✓ Final quantized model found
Size: 59.5 KB


In [ ]:
# === Export + automatic Drive backup ===
import os, json, shutil, datetime

tflite_src = (
    f"/content/trained_models/{OUTPUT_NAME}/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)
assert os.path.exists(tflite_src), f"No model at {tflite_src}"

OUT_TFLITE = f"/content/{OUTPUT_NAME}.tflite"
OUT_JSON = f"/content/{OUTPUT_NAME}.json"

shutil.copy2(tflite_src, OUT_TFLITE)

manifest = {
    "type": "micro",
    "wake_word": WAKE_WORD,
    "author": AUTHOR,
    "model": f"{OUTPUT_NAME}.tflite",
    "trained_languages": TRAINED_LANGUAGES,
    "version": 3,
    "micro": {
        "probability_cutoff": PROBABILITY_CUTOFF,
        "feature_step_size": 10,
        "sliding_window_size": SLIDING_WINDOW_SIZE,
        "tensor_arena_size": TENSOR_ARENA_SIZE,
        "minimum_esphome_version": "2024.7.0",
    },
}
if AUTHOR_WEBSITE.strip():
    manifest["website"] = AUTHOR_WEBSITE.strip()

with open(OUT_JSON, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"TFLite: {os.path.getsize(OUT_TFLITE)/1024:.1f} KB")
print(json.dumps(manifest, indent=2))

artifacts = [
    (OUT_TFLITE, f"{OUTPUT_NAME}.tflite"),
    (OUT_JSON, f"{OUTPUT_NAME}.json"),
    (
        f"/content/trained_models/{OUTPUT_NAME}/best_weights.weights.h5",
        f"{OUTPUT_NAME}_best_weights.weights.h5",
    ),
    (
        "/content/training_parameters.yaml",
        f"{OUTPUT_NAME}_training_parameters.yaml",
    ),
    (
        f"{DRIVE_DIR}/{REAL_SAMPLES_ZIP_NAME}",
        REAL_SAMPLES_ZIP_NAME,
    ),
]

for src_path, dest_name in artifacts:
    assert os.path.exists(src_path), f"Backup source missing: {src_path}"
    dest = f"{DRIVE_DIR}/{dest_name}"
    if os.path.abspath(src_path) != os.path.abspath(dest):
        shutil.copy2(src_path, dest)
    print(f"saved -> {dest}")

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f"{DRIVE_DIR}/_run_finished.txt", "w") as f:
    f.write(
        f"Nyra IT V3 training finished at {ts}\n"
        f"Wake word display name: {WAKE_WORD}\n"
        f"Pronunciation target: {PRONUNCIATION} (Italian)\n"
        f"Model ID: {OUTPUT_NAME}\n"
        f"Real positive weight: {REAL_POSITIVE_WEIGHT}\n"
        f"Synthetic positive weight: {SYNTHETIC_POSITIVE_WEIGHT}\n"
        f"Real train repetition: {REAL_TRAIN_REPETITION}\n"
        f"Synthetic train repetition: {SYNTHETIC_TRAIN_REPETITION}\n"
        f"Datasets: real_positives,synthetic_it_positives,"
        f"speech,no_speech,dinner_party,dinner_party_eval,confusables\n"
        f"Output: {OUTPUT_NAME}.tflite\n"
    )

print()
print("==============================================")
print("DONE — NYRA IT V3 IS SAVED TO GOOGLE DRIVE")
print("==============================================")
print(DRIVE_DIR)


TFLite: 59.5 KB
{
  "type": "micro",
  "wake_word": "Nyra",
  "author": "Nicola",
  "model": "nyra_it.tflite",
  "trained_languages": [
    "it"
  ],
  "version": 3,
  "micro": {
    "probability_cutoff": 0.7,
    "feature_step_size": 10,
    "sliding_window_size": 3,
    "tensor_arena_size": 50000,
    "minimum_esphome_version": "2024.7.0"
  }
}
saved -> /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it.tflite
saved -> /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it.json
saved -> /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it_best_weights.weights.h5
saved -> /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it_training_parameters.yaml
saved -> /content/drive/MyDrive/wakeword_training_nyra_it/nyra_it_real_samples.zip

DONE — NYRA IT V3 IS SAVED TO GOOGLE DRIVE
/content/drive/MyDrive/wakeword_training_nyra_it


In [ ]:
# === Final status / disk usage ===
import os

print("Drive artifacts:")
os.system(f'ls -lh "{DRIVE_DIR}"')

print()
print("Runtime disk:")
os.system("df -h /content | tail -1")


Drive artifacts:

Runtime disk:


0

## Deploy to ESPHome

After the run finishes, copy:

- `nyra_it.tflite`
- `nyra_it.json`

from:

`MyDrive/wakeword_training_nyra_it/`

to your ESPHome wake-word folder.

The visible wake-word name is **Nyra**.  
The target pronunciation is Italian **Nira / Nìra**.

This V3 model is trained with:
- the real Nyra positive recordings from `nyra_it_real_samples.zip`;
- native Italian Piper synthetic positives;
- a mandatory 30-sample synthetic pronunciation preview;
- synthetic Italian hard phonetic confusables;
- generic `speech`;
- generic `no_speech`;
- `dinner_party`;
- `dinner_party_eval` for ambient false-positive evaluation.

Initial manifest sensitivity:
- probability cutoff: **0.70**
- sliding window: **3**

Because Nyra/Nira is a short two-syllable wake word, first test false activations in normal daily use. If a recurring false trigger appears, collect that actual phrase later and use it as a targeted real negative in a future model version.
